In [1]:
from pprint import pprint
import json
import subprocess
import sys

import msgspec
from tqdm.auto import tqdm
import pandas as pd
from pathlib import Path
import os
import re
import torch
from sentence_transformers import models, SentenceTransformer
from transformers import AutoTokenizer, AutoModel, HfArgumentParser
from tevatron.retriever.arguments import DataArguments, ModelArguments
from tevatron.retriever.arguments import TevatronTrainingArguments as TrainingArguments
from transformers import HfArgumentParser
from tevatron.retriever.collator import EncodeCollator
from tevatron.retriever.dataset import EncodeDataset
from torch.utils.data import DataLoader
from tevatron.retriever.modeling.dense import DenseModel
from tevatron.retriever.modeling.encoder import EncoderOutput

encoder = msgspec.json.Encoder()
decoder = msgspec.json.Decoder()

# Utils


In [3]:
def read_json(file_path, jsonl=False):
    file_path = Path(file_path)
    if not file_path.is_file():
        raise ValueError("filepath is not a file")
    # if not file_path.suffix == ".jsonl" and jsonl:
    #     raise ValueError("the file is not jsonl")
    # if not file_path.suffix == ".json" and not jsonl:
    #     raise ValueError("the file is not json")
    file_path = file_path.__str__()

    output = []

    with open(file_path, "rb") as file:
        data = file.read()
        if jsonl:
            output = decoder.decode_lines(data)
        else:
            output = decoder.decode(data)

    print(f"The file is of type: {type(output)}")
    print(f"The file contains {len(output)} items.")
    return output


def write_json(file_path, data, jsonl=False):
    with open(file_path, "wb") as file:
        if jsonl:
            file.write(encoder.encode_lines(data))
        else:
            file.write(encoder.encode(data))
    print(f"The file contains {len(data)} items.")
    print("Saved to", file_path)


def read_tsv(file_path, row_names=None):
    file_path = Path(file_path)
    if file_path.is_file() and file_path.suffix == ".tsv":
        temp = pd.read_csv(file_path, sep="\t", names=row_names)
    else:
        raise ValueError("filepath is not a file or it is not a tsv file.")
    return temp


# Compute token embeddings
def get_sentence_embeddings(data_args, model, tokenizer, q=None, p=None):
    
    if q is not None:
        if not isinstance(q, list):
            q = [q]
        q = tokenizer(
            q,
            padding=True,
            truncation=True,
            max_length=(
                data_args.query_max_len - 1
                if data_args.append_eos_token
                else data_args.query_max_len
            ),
            pad_to_multiple_of=data_args.pad_to_multiple_of,
            return_attention_mask=True,
            return_tensors="pt",
            return_token_type_ids=False,
            add_special_tokens=True,
        )
        for k, v in q.items():
            q[k] = v.to("cuda")
    elif p is not None:
        if not isinstance(p, list):
            p = [p]
        p = tokenizer(
            p,
            padding=True,
            truncation=True,
            max_length=(
                data_args.passage_max_len - 1
                if data_args.append_eos_token
                else data_args.passage_max_len
            ),
            pad_to_multiple_of=data_args.pad_to_multiple_of,
            return_attention_mask=True,
            return_tensors="pt",
            return_token_type_ids=False,
            add_special_tokens=True,
        )
        for k, v in p.items():
            p[k] = v.to("cuda")
    
    model.eval()
    with torch.amp.autocast("cuda", dtype=torch.bfloat16):
        with torch.no_grad():
            if q is not None:
                output = model(query=q).q_reps
            else:
                output = model(passage=p).p_reps
    return output


def get_cosine_similarity(a, b, normalize=True):
    # This assume that the embedding has been normalized
    
    if normalize:
        a = torch.nn.functional.normalize(a, p=2, dim=1)
        b = torch.nn.functional.normalize(b, p=2, dim=1)
    
    return torch.matmul(a, b.T)

In [4]:
# bsz : batch size (number of positive pairs)
# d   : latent dim
# x   : Tensor, shape=[bsz, d]
#       latents for one side of positive pairs
# y   : Tensor, shape=[bsz, d]
#       latents for the other side of positive pairs

def align_loss(x, y, alpha=2):
    return (x - y).norm(p=2, dim=1).pow(alpha).mean()

def uniform_loss(x, t=2):
    return torch.pdist(x, p=2).pow(2).mul(-t).exp().mean().log()

def get_info(q_pos, q_neg, p, model_list, tokenizer, model_checkpoint_dir_list, data_args):
    
    for ix, (model, model_checkpoint_dir) in enumerate(zip(model_list, model_checkpoint_dir_list)):
        p1 = get_sentence_embeddings(data_args, model, tokenizer, p=p)
        q_pos1 = get_sentence_embeddings(data_args, model, tokenizer, q=q_pos)
        q_neg1 = get_sentence_embeddings(data_args, model, tokenizer, q=q_neg)
        
        print(f"\nModel{ix+1}:", Path(model_checkpoint_dir).stem)
        print(f"sim(p, q+): {get_cosine_similarity(p1, q_pos1).mean()}\nsim(p, q-): {get_cosine_similarity(p1, q_neg1).mean()}\nalign_loss(p, q+): {align_loss(p1, q_pos1).mean()}\nalign_loss(p, q-): {align_loss(p1, q_neg1).mean()}\ntemporal sim(p, q+): {get_cosine_similarity(p1[:, 512:], q_pos1[:, 512:]).mean()}\ntemporal sim(p, q-): {get_cosine_similarity(p1[:, 512:], q_neg1[:, 512:]).mean()}\nuniformity: {uniform_loss(torch.concat([p1, q_pos1, q_neg1], dim=0))}\n")
        # print(f"sim(p, q+): {get_cosine_similarity(p1, q_pos1)}\nsim(p, q-): {get_cosine_similarity(p1, q_neg1)}\nalign_loss(p, q+): {align_loss(p1, q_pos1)}\nalign_loss(p, q-): {align_loss(p1, q_neg1)}\ntemporal sim(p, q+): {get_cosine_similarity(p1[:, 512:], q_pos1[:, 512:])}\ntemporal sim(p, q-): {get_cosine_similarity(p1[:, 512:], q_neg1[:, 512:])}")
    
    # p2 = get_sentence_embeddings(data_args, model2, tokenizer, p=p)
    # q_pos2 = get_sentence_embeddings(data_args, model2, tokenizer, q=q_pos)
    # q_neg2 = get_sentence_embeddings(data_args, model2, tokenizer, q=q_neg)
    
    # print("Model2:", Path(model2_checkpoint_dir).stem)
    # # print(f"sim(p, q+): {get_cosine_similarity(p2, q_pos2).mean()}\nsim(p, q-): {get_cosine_similarity(p2, q_neg2). mean()}\nalign_loss(p, q+): {align_loss(p2, q_pos2).mean()}\nalign_loss(p, q-): {align_loss(p2, q_neg2).mean()}\ntemporal sim(p, q+): {get_cosine_similarity(p2[:, 512:], q_pos2[:, 512:]).mean()}\ntemporal sim(p, q-): {get_cosine_similarity(p2[:, 512:], q_neg2[:, 512:]).mean()}\nuniformity: {uniform_loss(torch.concat([p2, q_pos2, q_neg2], dim=0))}")
    # print(f"sim(p, q+): {get_cosine_similarity(p2, q_pos2)}\nsim(p, q-): {get_cosine_similarity(p2, q_neg2)}\nalign_loss(p, q+): {align_loss(p2, q_pos2)}\nalign_loss(p, q-): {align_loss(p2, q_neg2)}\ntemporal sim(p, q+): {get_cosine_similarity(p2[:, 512:], q_pos2[:, 512:])}\ntemporal sim(p, q-): {get_cosine_similarity(p2[:, 512:], q_neg2[:, 512:])}")
    

# Load models

In [ ]:
os.listdir("/home/thuy0050/mg61_scratch2/thuy0050/exp/tevatron/temporal_nobel_prize/ts-retriever/contriever/bs64")

In [ ]:
DATA_ROOT_DIR="/home/thuy0050/mg61_scratch2/thuy0050/data/third_work"
OUTPUT_DIR_ROOT="/home/thuy0050/mg61_scratch2/thuy0050/exp/tevatron"

DATA_NAME="temporal_nobel_prize"
MODEL_NAME="ts-retriever"
BACKBONE="bge-base-en-v1.5"
EXP_NAME = ["BAAI/bge-base-en-v1.5", "baseline", "semantic_matryoshka", "temporal", "temporal_projector", "temporal_projector_reconstruction", "temporal_projector_reconstruction_kl_loss"]
model_list = []
model_checkpoint_dir_list = []
torch_dtype = torch.bfloat16

for ix, exp in enumerate(EXP_NAME):
    
    if ix == 0:
        OUTPUT_DIR = exp
    else:
        OUTPUT_DIR=f"{OUTPUT_DIR_ROOT}/{DATA_NAME}/{MODEL_NAME}/{BACKBONE}/{exp}"
    CHECKPOINT_DIR=OUTPUT_DIR

    sys.argv = [
        "train_tsretriever_with_temporal_v4.py",  # dummy script name
        "--pooling", "avg",
        "--bf16",
        "--normalize",
        "--query_max_len", "512",
        "--passage_max_len", "512",
        "--attn_implementation", "sdpa",
        "--lora",
        "--lora_r", "4",
        "--lora_alpha", "16",
        "--lora_target_modules", "all-linear",
        "--modules_to_save", "temporal_projector",
        "--dataset_name", f"{DATA_ROOT_DIR}/tevatron/Tevatron___msmarco-passage",
        "--dataset_path", f"{DATA_ROOT_DIR}/temporal/temporal_nobel_prize/train/train_temporal_v2.jsonl",
        "--eval_dataset_path", f"{DATA_ROOT_DIR}/temporal/temporal_nobel_prize/train/dev.jsonl",
        "--model_name_or_path", CHECKPOINT_DIR,
        "--run_name", f"{BACKBONE}_{EXP_NAME}"
    ]


    parser = HfArgumentParser((ModelArguments, DataArguments, TrainingArguments))

    model_args, data_args, training_args = parser.parse_args_into_dataclasses()
    model_args: ModelArguments
    data_args: DataArguments
    training_args: TrainingArguments

    tokenizer = AutoTokenizer.from_pretrained(model_args.model_name_or_path)

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    if data_args.padding_side == 'right':
        tokenizer.padding_side = 'right'
    else:
        tokenizer.padding_side = 'left'
        
    model = DenseModel.load(
        model_args.model_name_or_path,
        pooling=model_args.pooling,
        normalize=model_args.normalize,
        lora_name_or_path=model_args.lora_name_or_path,
        cache_dir=model_args.cache_dir,
        torch_dtype=torch_dtype,
        attn_implementation=model_args.attn_implementation,
    )
    model = model.to("cuda")
    model_list.append(model)
    model_checkpoint_dir_list.append(CHECKPOINT_DIR)

# Analysis

In [ ]:
prompt = "Represent this sentence for searching relevant passages: "

dev = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/dev.jsonl", jsonl=True)

p_text = []
q_pos_text = []
q_neg_text = []
p_index = []
q_pos_count = []
q_neg_count = []

for ix, item in enumerate(dev):
    p_index.append(ix)
    p_text.append(item["query"])
    q_pos_count.append(len(item["positive_passages"]))
    q_neg_count.append(len(item["negative_passages"]))
    for i in item["positive_passages"]:
        q_pos_text.append(i['text'])
    for i in item["negative_passages"]:
        q_neg_text.append(i['text'])

# q_pos_text = prompt + "Jermaine Beckford played for which team from 2003 to 2004?"
# qt_pos_text = prompt + "from 2003 to 2004?"

# q_neg_text = prompt + "Jermaine Beckford played for which team from 1996 to 2002?"
# qt_neg_text = prompt + "from 1996 to 2002"

# p_text = "Beckford originally began his career in the Chelsea youth team , coming through the schoolboy ranks at the same time as Carlton Cole . Rejected by Chelsea in 2003 , he was signed up by Wealdstone , then in the Isthmian Premier League , and played as a semi-professional for three years whilst also working as a windscreen fitter for the RAC . His very impressive goal scoring record for Wealdstone attracted a lot of attention from Football League sides and reportedly more than 30 professional clubs showed an interest in the prolific striker , with many sending scouts to watch him play for Wealdstone . He had a trial with Championship side Crystal Palace , before signing for Leeds United in March 2006 for an undisclosed fee , having scored 35 goals in 40 games for Wealdstone that season ."

In [ ]:
def get_alignment_score(q_pos, p, q_pos_count, p_index):
    align_score = 0.0
    temp = 0
    for k, v in zip(p_index, q_pos_count):
        align_score += align_loss(q_pos[temp:temp+v], p[k])
        temp += v
    return align_score / len(p_index)

def get_uniformity_score(p, q_pos, q_neg):
    return uniform_loss(torch.concat([p, q_pos, q_neg], dim=0))

## Test case

In [ ]:
ix = 0
p_text[ix], q_pos_text[ix:ix+q_pos_count[ix]], q_neg_text[ix:ix+q_neg_count[ix]]
temp_p_text = "He signed a two-year contract with League One side Crewe Alexandra in June 2013 after manager Steve Davis paid Macclesfield an undisclosed fee .\
However , he played just five games for the Railwaymen , being sent on two loan spells to Lincoln City , before returning to Macclesfield Town in February 2015 .\
He then played for newly relegated Notts County in League Two for two seasons .\
In July 2017 , Audel joined Barrow , moving to Welling United a year later ."
print(temp_p_text.split("."))

In [ ]:
for i, (model, model_checkpoint_dir) in enumerate(zip(model_list, model_checkpoint_dir_list)):
    p1 = get_sentence_embeddings(data_args, model, tokenizer, p=temp_p_text)[:, 512:]
    sentence_list = q_pos_text[ix:ix+q_pos_count[ix]] + q_neg_text[ix:ix+q_neg_count[ix]]
    q_pos1 = get_sentence_embeddings(data_args, model_list[0], tokenizer, q=sentence_list)[:, 512:]
    print(f"\nModel{i+1}:", Path(model_checkpoint_dir).stem)
    print("Passage:", temp_p_text)
    for sent, sim in zip(sentence_list, get_cosine_similarity(p1, q_pos1).cpu().numpy().tolist()[0]):
        print(f"{sent}: {sim:.4f}")

### Controlled experiments

In [ ]:
for ix, (model, model_checkpoint_dir) in enumerate(zip(model_list, model_checkpoint_dir_list)):
    p1 = get_sentence_embeddings(data_args, model, tokenizer, p=temp_p_text.split(".")[0])
    sentence_list = [
        f"Thierry Audel played for which team before {x}" for x in [str(i) for i in range(2013-5, 2013+5+1)]
    ]
    q_pos1 = get_sentence_embeddings(data_args, model_list[0], tokenizer, q=sentence_list)
    print(f"\nModel{ix+1}:", Path(model_checkpoint_dir).stem)
    print("Passage:", temp_p_text.split(".")[0])
    for sent, sim in zip(sentence_list, get_cosine_similarity(p1, q_pos1).cpu().numpy().tolist()[0]):
        print(f"{sent}: {sim:.4f}")

### Get all the scores given p_list, q+_list, q-_list for all models

In [ ]:
get_info(q_pos_text[ix:ix+q_pos_count[ix]], q_neg_text[ix:ix+q_neg_count[ix]], p_text[ix], model_list, tokenizer, model_checkpoint_dir_list, data_args)

### To get the uniformity scores correctly!

In [ ]:
for ix, (model, model_checkpoint_dir) in enumerate(zip(model_list, model_checkpoint_dir_list)):
        print(f"\nModel{ix+1}:", Path(model_checkpoint_dir).stem)
        p1 = get_sentence_embeddings(data_args, model, tokenizer, p=p_text)
        q_pos1 = get_sentence_embeddings(data_args, model, tokenizer, q=q_pos_text)
        q_neg1 = get_sentence_embeddings(data_args, model, tokenizer, q=q_neg_text)
        
        print("Alignment score:", get_alignment_score(q_pos1, p1, q_pos_count, p_index).cpu().item())
        print("Uniformity score:", get_uniformity_score(q_pos1, p1, q_neg1).cpu().item())

In [ ]:
temp = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/implicit_event_based/ComplexTempQA/ComplexTempQA_small.json", jsonl=True)

In [ ]:
temp[0]

In [ ]:
temp[1]

# Test bge-multilingual-gemma2

In [ ]:
import torch
import torch.nn.functional as F

from torch import Tensor
from transformers import AutoTokenizer, AutoModel


def last_token_pool(last_hidden_states: Tensor,
                 attention_mask: Tensor) -> Tensor:
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    else:
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = last_hidden_states.shape[0]
        return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]


def get_detailed_instruct(task_description: str, query: str) -> str:
    return f'<instruct>{task_description}\n<query>{query}'


task = 'Given a web search query, retrieve relevant passages that answer the query.'
queries = [
    get_detailed_instruct(task, 'how much protein should a female eat'),
    get_detailed_instruct(task, 'summit define')
]
# No need to add instructions for documents
documents = [
    "As a general guideline, the CDC's average requirement of protein for women ages 19 to 70 is 46 grams per day. But, as you can see from this chart, you'll need to increase that if you're expecting or training for a marathon. Check out the chart below to see how much protein you should be eating each day.",
    "Definition of summit for English Language Learners. : 1  the highest point of a mountain : the top of a mountain. : 2  the highest level. : 3  a meeting or series of meetings between the leaders of two or more governments."
]
input_texts = queries + documents

tokenizer = AutoTokenizer.from_pretrained('BAAI/bge-multilingual-gemma2')
model = AutoModel.from_pretrained('BAAI/bge-multilingual-gemma2')
model.eval()

max_length = 4096
# Tokenize the input texts
batch_dict = tokenizer(input_texts, max_length=max_length, padding=True, truncation=True, return_tensors='pt', pad_to_multiple_of=8)

with torch.no_grad():
    outputs = model(**batch_dict)
    embeddings = last_token_pool(outputs.last_hidden_state, batch_dict['attention_mask'])
    
# normalize embeddings
embeddings = F.normalize(embeddings, p=2, dim=1)
scores = (embeddings[:2] @ embeddings[2:].T) * 100
print(scores.tolist())
# [[55.92064666748047, 1.6549524068832397], [-0.2698777914047241, 49.95653533935547]]


In [ ]:
queries = [get_detailed_instruct(task, x) for x in [
        "Thierry Audel played for which team after 2013?", 
        "Thierry Audel played for which team before 2013?", 
        "Thierry Audel played for which team in 2013?",
        "Thierry Audel played for which team as of 2013?",
        "Thierry Audel played for which team during 2013?",
    ]
]
documents = [
    f"He signed a two-year contract with League One side Crewe Alexandra in June {x} after manager Steve Davis paid Macclesfield an undisclosed fee" for x in [str(i) for i in range(2013-5, 2013+5+1)]
]

input_texts = queries + documents

max_length = 4096
# Tokenize the input texts
batch_dict = tokenizer(input_texts, max_length=max_length, padding=True, truncation=True, return_tensors='pt', pad_to_multiple_of=8)
model.cuda()

with torch.no_grad():
    outputs = model(**batch_dict.to("cuda"))
    embeddings = last_token_pool(outputs.last_hidden_state, batch_dict['attention_mask'])
    
# normalize embeddings
embeddings = F.normalize(embeddings, p=2, dim=1)

In [ ]:
input_texts

In [ ]:
get_cosine_similarity(embeddings[:5, :], embeddings[5:, :]).cpu().numpy().tolist()

In [ ]:
sum([v.numel() for k, v in model.named_parameters()]) / 1024**2

# Dev

In [5]:
configs = {
    "bge": dict(
        checkpoint="BAAI/bge-base-en-v1.5",
        pooling="cls",
        query_prefix="'Represent this sentence for searching relevant passages: '",
        passage_prefix="''",
        query_prompts="'Represent this sentence for searching relevant passages: '",
        corpus_prompts="''",
        normalize="--normalize",
        padding_side="right",
        matryoshka_dim="768",
        temperature="0.02",
    ),
    "bgem3": dict(
        checkpoint="BAAI/bge-m3",
        pooling="cls",
        query_prefix="''",
        passage_prefix="''",
        query_prompts="''",
        corpus_prompts="''",
        normalize="--normalize",
        padding_side="right",
        matryoshka_dim="1024",
        temperature="0.02",
    ),
    "contriever": dict(
        checkpoint="facebook/contriever",
        pooling="mean",
        query_prefix="''",
        passage_prefix="''",
        query_prompts="''",
        corpus_prompts="''",
        normalize="",
        padding_side="right",
        matryoshka_dim="768",
        temperature="0.05",
    ),
    "gte": dict(
        checkpoint="thenlper/gte-base",
        pooling="mean",
        query_prefix="''",
        passage_prefix="''",
        query_prompts="''",
        corpus_prompts="''",
        normalize="--normalize",
        padding_side="right",
        matryoshka_dim="768",
        temperature="0.02",
    ),
    "gte1.5": dict(
        checkpoint="Alibaba-NLP/gte-base-en-v1.5",
        pooling="cls",
        query_prefix="''",
        passage_prefix="''",
        query_prompts="''",
        corpus_prompts="''",
        normalize="--normalize",
        padding_side="right",
        matryoshka_dim="768",
        matryoshka_dim_list=[768],
        temperature="0.01",
    ),
    "nomic": dict(
        checkpoint="nomic-ai/nomic-embed-text-v1.5",
        pooling="mean",
        query_prefix="search_query:",
        passage_prefix="search_document:",
        query_prompts="search_query:",
        corpus_prompts="search_document:",
        normalize="--normalize",
        padding_side="right",
        matryoshka_dim="768",
        temperature="0.02",
    ),
    "qwen3": dict(
        checkpoint="Qwen/Qwen3-Embedding-0.6B",
        pooling="last",
        query_prefix="'Instruct: Given a web search query, retrieve relevant passages that answer the query\\nQuery:'",
        passage_prefix="''",
        query_prompts="'Instruct: Given a web search query, retrieve relevant passages that answer the query\\nQuery:'",
        corpus_prompts="''",
        normalize="--normalize",
        padding_side="left",
        matryoshka_dim="1024",
        temperature="0.02",
    ),
}

import torch, faiss
import numpy as np
import pickle

from collections import defaultdict

qrel_dict = defaultdict(list)
with open(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/test/qrel.txt",
    "r",
) as f:
    for line in f:
        qid, _, docid, _ = line.split(" ")
        qrel_dict[int(qid)].append(int(docid))
        
corpus_jsonl = read_json(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/test/corpus.jsonl",
    jsonl=True,
)
corpus_jsonl_docid = [x["docid"] for x in corpus_jsonl]
corpus_jsonl_text = [x["text"] for x in corpus_jsonl]
corpus_docid_to_text_dict = {
    _id: text for _id, text in zip(corpus_jsonl_docid, corpus_jsonl_text)
}

The file is of type: <class 'list'>
The file contains 989 items.


## GTE

In [ ]:
import os
from tevatron.retriever.modeling.dense import DenseModel

def get_model_and_tokenizer(
    cfg, model_name_or_path, 
    matryoshka_dim=768,
    batch_size=128,
    num_workers=4,
):

    # Example: directly supply your arguments as dicts
    arg_dict = {
        "model_name_or_path": model_name_or_path,
        "tokenizer_name": None,
        "padding_side": cfg["padding_side"],
        "dataset_name": "LouisDo2108/temporal-nobel-prize",
        "dataset_config": "corpus",
        "dataset_path": "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/test/query.jsonl",
        "encode_is_query": True,
        "lora_name_or_path": model_name_or_path,
        "pooling": cfg['pooling'],
        "normalize": True,
        "bf16": True,
    }

    model_args, data_args, training_args = parser.parse_dict(
        {**arg_dict}
    )
    tokenizer = AutoTokenizer.from_pretrained(
        model_args.tokenizer_name if model_args.tokenizer_name else model_args.model_name_or_path,
        cache_dir=model_args.cache_dir
    )
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    if data_args.padding_side == 'right':
        tokenizer.padding_side = 'right'
    else:
        tokenizer.padding_side = 'left'

    if training_args.bf16:
        torch_dtype = torch.bfloat16
    elif training_args.fp16:
        torch_dtype = torch.float16
    else:
        torch_dtype = torch.float32

    model = DenseModel.load(
        model_args.model_name_or_path,
        pooling=model_args.pooling,
        normalize=model_args.normalize,
        lora_name_or_path=model_args.lora_name_or_path,
        cache_dir=model_args.cache_dir,
        torch_dtype=torch_dtype,
        attn_implementation=model_args.attn_implementation,
    )
    model.matryoshka_dim = matryoshka_dim
    
    # encode_dataset = EncodeDataset(
    #     data_args=data_args,
    # )

    # encode_collator = EncodeCollator(
    #     data_args=data_args,
    #     tokenizer=tokenizer,
    # )

    # encode_loader = DataLoader(
    #     encode_dataset,
    #     batch_size=batch_size,
    #     collate_fn=encode_collator,
    #     shuffle=False,
    #     drop_last=False,
    #     num_workers=num_workers,
    # )
    
    model = model.to("cuda")
    model = model.eval()
    return model, tokenizer, model_args, data_args, training_args

def _pooling(last_hidden_state, attention_mask, pooling="cls", normalize=True):
    if pooling in ['cls', 'first']:
        reps = last_hidden_state[:, 0]
    elif pooling in ['mean', 'avg', 'average']:
        masked_hiddens = last_hidden_state.masked_fill(~attention_mask[..., None].bool(), 0.0)
        reps = masked_hiddens.sum(dim=1) / attention_mask.sum(dim=1)[..., None]
    elif pooling in ['last', 'eos']:
        left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
        if left_padding:
            reps = last_hidden_state[:, -1]
        else:
            sequence_lengths = attention_mask.sum(dim=1) - 1
            batch_size = last_hidden_state.shape[0]
            reps = last_hidden_state[torch.arange(batch_size, device=last_hidden_state.device), sequence_lengths]
    else:
        raise ValueError(f'unknown pooling method: {self.pooling}')

        # if self.encoder.eval() and self.matryoshka_dim is not None:
        #     print(f"Eval with matryoshka dim {self.matryoshka_dim}")
        #     reps = reps[:, :self.matryoshka_dim]

    if normalize:
        reps = torch.nn.functional.normalize(reps, p=2, dim=-1)
    return reps

def get_embedding(tokenizer, query, passage, model_args, data_args, pooling=True):
    encoded = []
    lookup_indices = []
    
    data_args.passage_prefix = data_args.passage_prefix.replace("\\n", "\n").strip()
    if data_args.passage_prefix != "":
        data_args.passage_prefix = data_args.passage_prefix + " "
        
    data_args.query_prefix = data_args.query_prefix.replace("\\n", "\n").strip()
    if data_args.query_prefix != "":
        data_args.query_prefix = data_args.query_prefix + " "

    query = data_args.query_prefix + query
    passage = data_args.passage_prefix + passage
    passages = [query, passage]
    inputs = tokenizer(
        passages,
        padding=True,
        truncation=True,
        max_length=512,
        pad_to_multiple_of=8,
        return_attention_mask=True,
        return_tensors="pt",
        return_token_type_ids=False,
        add_special_tokens=True,
        padding_side="right",
        return_offsets_mapping=True,
        
    )
    inputs = {k: v.to("cuda") for k, v in inputs.items()}

    # for iix, x in enumerate(inputs["offset_mapping"]):
    #     for ix, (i, j) in enumerate(x.cpu().numpy().tolist()):
    #         if i == j:
    #             continue
    #         print(f"{ix}, {i}-{j}: {passages[iix][i:j]}")
        
    query_mapping, passage_mapping = inputs["offset_mapping"][0], inputs["offset_mapping"][1]
    inputs.pop("offset_mapping")

    with (
        torch.autocast(
            "cuda", dtype=torch.float16 if training_args.fp16 else torch.bfloat16
        )
        if training_args.fp16 or training_args.bf16
        else nullcontext()
    ):
        with torch.no_grad():
            model_output: EncoderOutput = model.encode_query(inputs, pooling=False)
            pooled_output = model.encode_query(inputs, pooling=True)
            
    pooled_output = _pooling(model_output, inputs['attention_mask'], pooling=model_args.pooling, normalize=model_args.normalize)
            
    return model_output, pooled_output, inputs, query_mapping, passage_mapping


# Maxsim operator:
def maxsim(a, b, normalize=True):
    # This assume that the embedding has been normalized
    
    if normalize:
        a = torch.nn.functional.normalize(a, p=2, dim=1)
        b = torch.nn.functional.normalize(b, p=2, dim=1)
    
    return torch.max(torch.matmul(a, b.T), dim=1)


# Maxsim operator:
def topk(a, b, normalize=True, k=10):
    # This assume that the embedding has been normalized
    
    if normalize:
        a = torch.nn.functional.normalize(a, p=2, dim=1)
        b = torch.nn.functional.normalize(b, p=2, dim=1)
        
    
    
    return torch.topk(torch.matmul(a, b.T), dim=1, k=k)


def show_topk_tokens(pooled_output, model_output, inputs, query_mapping, query, tokenizer, k=10):
    
    query_attention_mask = inputs['attention_mask'][0, :, None]
    passage_attention_mask = inputs['attention_mask'][1, :, None]
    
    query_input_ids = inputs['input_ids'][0, :]
    passage_input_ids = inputs['input_ids'][1, :]
    
    query_pooled_embedding = pooled_output[0]
    passage_pooled_embedding = pooled_output[1]
    
    query_model_output = model_output[0, :, :]
    passage_model_output = model_output[1, :, :]

    for dim in [64, 128, 256, 512, 768]:
        # Attend the pooled passage embedding with all token embeddings of the query
        v, idx = topk(
            passage_pooled_embedding[None, :dim], query_model_output[:, :dim] * query_attention_mask, k=k
        )
        idx = idx.cpu().numpy().tolist()[0]
        values = [round(x, 2) for x in v.cpu().numpy().tolist()[0]]
        
        texts = []
        for ix, i in enumerate(idx):
            if query_input_ids[i] == tokenizer.eos_token_id:
                corresponding_text = "<EOS>"
            elif query_input_ids[i] == tokenizer.cls_token_id:
                corresponding_text = "<CLS>"
            elif query_input_ids[i] == tokenizer.pad_token_id:
                corresponding_text = "<PAD>"
            else:
                corresponding_text = query[
                    query_mapping[i][0]:query_mapping[i][1]
                ]
            texts.append(corresponding_text)
        print(f"Topk for dim {dim}: {texts}")
        print(f"Topk scores: {values}")

       
if __name__ == "__main__":
    
    parser = HfArgumentParser((ModelArguments, DataArguments, TrainingArguments))
    
    # model_args, data_args, training_args = parser.parse_args_into_dataclasses()

    model_name = "gte1.5"
    # original_model_name = "thenlper/gte-base"
    original_model_name = "Alibaba-NLP/gte-base-en-v1.5"
    root_path = "/home/thuy0050/mg61_scratch2/thuy0050/exp/tevatron/temporal_nobel_prize/temporal"

    cfg = configs[model_name]
    # model_name_or_path = os.path.join(root_path, original_model_name, "matryoshka_64-128-256-512-768_filtered")
    model_name_or_path = original_model_name

    model, tokenizer, model_args, data_args, training_args = get_model_and_tokenizer(cfg,  model_name_or_path)
    
    query = "Who won the Nobel Prize in Medicine between 1946 and 1947?" # n x D embeddings
    passage = 'Bernardo Houssay\nThe Nobel Prize in Medicine 1947.\nBorn: 10, Apr, 1887, Buenos Aires, Argentina\nDied: 21, Sep, 1971, Buenos Aires, Argentina\nPrize motivation: "for his discovery of the part played by the hormone of the anterior pituitary lobe in the metabolism of sugar"\nAffiliation at the time of the award: Instituto de Biologia y Medicina Experimental (Institute for Biology and Experimental Medicine), Buenos Aires, Argentina\nPrize share: 1/2' # CLS token, D embedding
    
    # Top-k Sim operator: ColBERT, late-interactions 
    
    model_output, pooled_output, inputs, query_mapping, passage_mapping = get_embedding(tokenizer, query, passage, model_args, data_args, pooling=True)
    
    show_topk_tokens(pooled_output, model_output, inputs, query_mapping, query, tokenizer, k=10)

Either your model is not a PEFT-model or you are missing the lora_name_or_path argument.


Eval with matryoshka dim 768
Topk for dim 64: ['<CLS>', 'Medicine', 'in', '?', 'the', '1947', '1946', 'Who', 'won', 'and']
Topk scores: [1.0, 0.92, 0.89, 0.89, 0.89, 0.89, 0.89, 0.88, 0.87, 0.85]
Topk for dim 128: ['<CLS>', 'Medicine', 'in', '?', '1946', 'Who', '1947', 'the', '', 'and']
Topk scores: [1.0, 0.89, 0.87, 0.87, 0.87, 0.85, 0.85, 0.84, 0.82, 0.82]
Topk for dim 256: ['<CLS>', '?', 'Medicine', '1946', '1947', 'in', 'Who', 'the', '', 'and']
Topk scores: [1.0, 0.87, 0.87, 0.86, 0.85, 0.85, 0.84, 0.83, 0.79, 0.79]
Topk for dim 512: ['<CLS>', 'Medicine', '?', '1946', '1947', 'in', 'Who', 'the', '', 'won']
Topk scores: [1.0, 0.87, 0.87, 0.87, 0.86, 0.86, 0.85, 0.84, 0.82, 0.81]
Topk for dim 768: ['<CLS>', '1946', 'Medicine', '?', '1947', 'in', 'Who', 'the', 'Nobel', 'won']
Topk scores: [1.0, 0.88, 0.88, 0.87, 0.87, 0.86, 0.85, 0.85, 0.82, 0.82]


In [69]:
if __name__ == "__main__":
    
    parser = HfArgumentParser((ModelArguments, DataArguments, TrainingArguments))

    model_name = "gte1.5"
    original_model_name = "Alibaba-NLP/gte-base-en-v1.5"
    root_path = "/home/thuy0050/mg61_scratch2/thuy0050/exp/tevatron/temporal_nobel_prize/temporal"

    cfg = configs[model_name]
    model_name_or_path = os.path.join(root_path, original_model_name, "matryoshka_64-128-256-512-768_filtered")

    model, tokenizer, model_args, data_args, training_args = get_model_and_tokenizer(cfg,  model_name_or_path)
    
    query = "Who won the Nobel Prize in Medicine between 1946 and 1947?" # n x D embeddings
    passage = "1946 and 1947?" # 'Bernardo Houssay\nThe Nobel Prize in Medicine 1947.\nBorn: 10, Apr, 1887, Buenos Aires, Argentina\nDied: 21, Sep, 1971, Buenos Aires, Argentina\nPrize motivation: "for his discovery of the part played by the hormone of the anterior pituitary lobe in the metabolism of sugar"\nAffiliation at the time of the award: Instituto de Biologia y Medicina Experimental (Institute for Biology and Experimental Medicine), Buenos Aires, Argentina\nPrize share: 1/2' # CLS token, D embedding
    
    # Top-k Sim operator: ColBERT, late-interactions 
    
    model_output, pooled_output, inputs, query_mapping, passage_mapping = get_embedding(tokenizer, query, passage, model_args, data_args, pooling=True)
    
    show_topk_tokens(pooled_output, model_output, inputs, query_mapping, query, tokenizer, k=10)

Eval with matryoshka dim 768
Topk for dim 64: ['1947', '1946', '<CLS>', '?', 'Medicine', 'and', 'in', 'the', 'between', 'Nobel']
Topk scores: [0.7, 0.69, 0.64, 0.57, 0.56, 0.55, 0.55, 0.54, 0.53, 0.52]
Topk for dim 128: ['1946', '1947', '<CLS>', '?', 'and', 'the', 'Prize', 'between', 'Medicine', 'Who']
Topk scores: [0.66, 0.65, 0.64, 0.57, 0.55, 0.53, 0.5, 0.5, 0.49, 0.49]
Topk for dim 256: ['1946', '1947', '<CLS>', '?', 'and', 'between', 'the', 'Medicine', 'Who', '']
Topk scores: [0.65, 0.65, 0.62, 0.52, 0.5, 0.47, 0.46, 0.46, 0.45, 0.44]
Topk for dim 512: ['1947', '1946', '<CLS>', '?', 'and', 'between', 'Medicine', 'Who', 'the', '']
Topk scores: [0.66, 0.65, 0.63, 0.52, 0.49, 0.47, 0.47, 0.46, 0.46, 0.45]
Topk for dim 768: ['1947', '1946', '<CLS>', '?', 'Medicine', '', 'and', 'the', 'between', 'Who']
Topk scores: [0.73, 0.72, 0.69, 0.62, 0.59, 0.58, 0.58, 0.57, 0.56, 0.56]


In [68]:
if __name__ == "__main__":
    
    parser = HfArgumentParser((ModelArguments, DataArguments, TrainingArguments))

    model_name = "gte1.5"
    original_model_name = "Alibaba-NLP/gte-base-en-v1.5"
    root_path = "/home/thuy0050/mg61_scratch2/thuy0050/exp/tevatron/temporal_nobel_prize/temporal"

    cfg = configs[model_name]
    model_name_or_path = os.path.join(root_path, original_model_name, "qt-pt0.1_filtered")

    model, tokenizer, model_args, data_args, training_args = get_model_and_tokenizer(cfg,  model_name_or_path)
    
    query = "Who won the Nobel Prize in Medicine between 1946 and 1947?" # n x D embeddings
    passage = "1946 and 1947?" # 'Bernardo Houssay\nThe Nobel Prize in Medicine 1947.\nBorn: 10, Apr, 1887, Buenos Aires, Argentina\nDied: 21, Sep, 1971, Buenos Aires, Argentina\nPrize motivation: "for his discovery of the part played by the hormone of the anterior pituitary lobe in the metabolism of sugar"\nAffiliation at the time of the award: Instituto de Biologia y Medicina Experimental (Institute for Biology and Experimental Medicine), Buenos Aires, Argentina\nPrize share: 1/2' # CLS token, D embedding
    
    # Top-k Sim operator: ColBERT, late-interactions 
    
    model_output, pooled_output, inputs, query_mapping, passage_mapping = get_embedding(tokenizer, query, passage, model_args, data_args, pooling=True)
    
    show_topk_tokens(pooled_output, model_output, inputs, query_mapping, query, tokenizer, k=10)

Eval with matryoshka dim 768
Topk for dim 64: ['1947', '1946', '<CLS>', '?', 'the', 'and', '', 'between', 'Who', 'in']
Topk scores: [0.86, 0.86, 0.84, 0.84, 0.82, 0.82, 0.82, 0.81, 0.81, 0.81]
Topk for dim 128: ['1946', '1947', '<CLS>', '?', 'the', 'and', 'Who', '', 'won', 'between']
Topk scores: [0.82, 0.82, 0.81, 0.79, 0.78, 0.77, 0.77, 0.75, 0.75, 0.75]
Topk for dim 256: ['1946', '1947', '<CLS>', '?', 'and', 'between', 'the', 'Who', '', 'won']
Topk scores: [0.77, 0.77, 0.76, 0.72, 0.7, 0.69, 0.69, 0.68, 0.67, 0.67]
Topk for dim 512: ['1947', '1946', '<CLS>', '?', 'and', 'between', 'Who', 'the', '', 'won']
Topk scores: [0.75, 0.75, 0.74, 0.7, 0.68, 0.67, 0.65, 0.65, 0.65, 0.63]
Topk for dim 768: ['1947', '1946', '<CLS>', '?', 'and', 'between', 'the', '', 'Who', 'Medicine']
Topk scores: [0.79, 0.78, 0.77, 0.74, 0.73, 0.72, 0.71, 0.71, 0.71, 0.7]


In [73]:
if __name__ == "__main__":
    
    parser = HfArgumentParser((ModelArguments, DataArguments, TrainingArguments))

    model_name = "gte1.5"
    original_model_name = "Alibaba-NLP/gte-base-en-v1.5"
    root_path = "/home/thuy0050/mg61_scratch2/thuy0050/exp/tevatron/temporal_nobel_prize/temporal"

    cfg = configs[model_name]
    model_name_or_path = os.path.join(root_path, original_model_name, "dev")

    model, tokenizer, model_args, data_args, training_args = get_model_and_tokenizer(cfg,  model_name_or_path)
    
    query = "Who won the Nobel Prize in Medicine between 1946 and 1947?" # n x D embeddings
    passage = 'Bernardo Houssay\nThe Nobel Prize in Medicine 1947.\nBorn: 10, Apr, 1887, Buenos Aires, Argentina\nDied: 21, Sep, 1971, Buenos Aires, Argentina\nPrize motivation: "for his discovery of the part played by the hormone of the anterior pituitary lobe in the metabolism of sugar"\nAffiliation at the time of the award: Instituto de Biologia y Medicina Experimental (Institute for Biology and Experimental Medicine), Buenos Aires, Argentina\nPrize share: 1/2' # CLS token, D embedding
    
    # Top-k Sim operator: ColBERT, late-interactions 
    
    model_output, pooled_output, inputs, query_mapping, passage_mapping = get_embedding(tokenizer, query, passage, model_args, data_args, pooling=True)
    
    show_topk_tokens(pooled_output, model_output, inputs, query_mapping, query, tokenizer, k=10)

Eval with matryoshka dim 768
Topk for dim 64: ['<CLS>', 'the', 'in', 'won', 'Nobel', 'Who', '?', 'Prize', 'and', 'between']
Topk scores: [1.0, 0.99, 0.99, 0.99, 0.99, 0.99, 0.99, 0.99, 0.99, 0.99]
Topk for dim 128: ['<CLS>', 'the', 'won', 'in', '?', 'and', '1946', 'between', 'Who', 'Prize']
Topk scores: [1.0, 0.98, 0.98, 0.98, 0.98, 0.98, 0.98, 0.98, 0.97, 0.97]
Topk for dim 256: ['<CLS>', 'the', 'won', '?', 'in', '1946', 'Who', '1947', 'and', 'between']
Topk scores: [1.0, 0.97, 0.97, 0.97, 0.97, 0.96, 0.96, 0.96, 0.96, 0.96]
Topk for dim 512: ['<CLS>', 'the', '?', 'won', 'Who', 'in', '1946', '1947', 'and', 'between']
Topk scores: [1.0, 0.96, 0.95, 0.95, 0.95, 0.95, 0.95, 0.95, 0.95, 0.95]
Topk for dim 768: ['<CLS>', 'the', 'in', 'Who', 'won', '1946', '?', '1947', 'and', 'between']
Topk scores: [1.0, 0.96, 0.95, 0.95, 0.95, 0.95, 0.95, 0.95, 0.94, 0.94]


for all dimesions -> Desirable to have temporal tokens at the top-k positions (similarity) 
-> Work well for mean pooling -> Dilutes the scenario of missing of temporal info 

Apply to CLS pooling
-> Changing the semantic of the CLS token 
Lower dimensions -> Work well
Distort the semantic of the original CLS token

Example:
1. Query with temporal info
2. Query without temporal info

[temporal (64)|semantic] = 768
1. [temporal (64)] = 64 -> Works really well 
1. [temporal (64) | semantic (64)] = 128
...
[temporal(64)|semantic(704)] = 768



In [ ]:
# temporal_projector(temporal_tokens) -> Mean pooling, they average for all tokens 
# "Who won the Nobel Prize in Medicine between 1946 and 1947?" -> nxD-dimensional embedding
# t = temporal_projector("between 1946 and 1947") -> T-dimensional embedding
# align t with [T-dimensional embedding:] of the D-dimensional embedding 
# -> This works for mean pooling, since we are aligning with several tokens.

# gte-base: mean-pooling
# gte-base-v1.5: CLS-pooling

# CLS pooling -> (n+1)xD-dimensional embedding (first token) 
# -> Select the first token 
# t = temporal_projector([CLS]+"between 1946 and 1947") -> T-dimensional embedding

# embedding model + LoRA 
# (+ temporal projector aligns the first few sub-embeddings, discard after training)

# Mean pooling

In [420]:
data_args.encode_is_query = True
for (batch_ids, batch) in tqdm(encode_loader):
        lookup_indices.extend(batch_ids)
        with (
            torch.autocast(
                "cuda", dtype=torch.float16 if training_args.fp16 else torch.bfloat16
            )
            if training_args.fp16 or training_args.bf16
            else nullcontext()
        ):
            with torch.no_grad():
                for k, v in batch.items():
                    batch[k] = v.to("cuda")
                if data_args.encode_is_query:
                    model_output: EncoderOutput = model(query=batch)
                    encoded.append(model_output.q_reps.cpu().detach().numpy())
                else:
                    model_output: EncoderOutput = model(passage=batch)
                    encoded.append(model_output.p_reps.cpu().detach().numpy())
                    
with open(os.path.join(model_name_or_path, "corpus_emb_768.pkl"), "rb") as f:
    p_vecs, corpus_lookup_indices = pickle.load(f)

# create a basic flat index with dimension match our embedding
index = faiss.IndexFlatIP(len(p_vecs[0]))
# make sure the embeddings are float32
p_vecs = np.asarray(p_vecs, dtype=np.float32)
# use gpu to accelerate index searching
if torch.cuda.is_available():
    co = faiss.GpuMultipleClonerOptions()
    co.shard = True
    co.useFloat16 = True
    index = faiss.index_cpu_to_all_gpus(index, co=co)
# add all the embeddings to the index
index.add(p_vecs)

  0%|                                                                                                                  | 0/26 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokeniz

Eval with matryoshka dim 768
Eval with matryoshka dim 768
Eval with matryoshka dim 768
Eval with matryoshka dim 768
Eval with matryoshka dim 768
Eval with matryoshka dim 768
Eval with matryoshka dim 768
Eval with matryoshka dim 768


 50%|████████████████████████████████████████████████████▌                                                    | 13/26 [00:03<00:01,  7.41it/s]

Eval with matryoshka dim 768
Eval with matryoshka dim 768
Eval with matryoshka dim 768
Eval with matryoshka dim 768
Eval with matryoshka dim 768
Eval with matryoshka dim 768
Eval with matryoshka dim 768
Eval with matryoshka dim 768


 81%|████████████████████████████████████████████████████████████████████████████████████▊                    | 21/26 [00:03<00:00, 13.81it/s]

Eval with matryoshka dim 768
Eval with matryoshka dim 768
Eval with matryoshka dim 768
Eval with matryoshka dim 768
Eval with matryoshka dim 768
Eval with matryoshka dim 768
Eval with matryoshka dim 768
Eval with matryoshka dim 768


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 26/26 [00:03<00:00,  7.10it/s]


Eval with matryoshka dim 768
Eval with matryoshka dim 768


In [421]:
converted = []
scores = []
for batch in encoded:
    score, ids = index.search(
        batch, k=20
    )
    # convert the auto ids back to ids in the original dataset
    for s, ranked_list in zip(score, ids): 
        converted.append([corpus_lookup_indices[i] for i in ranked_list])
        scores.append(s)
        
count = 0
for i in range(len(converted)):
    qrel_set = set(qrel_dict[i])
    if not qrel_set.issubset(set(converted[i][:10])):
        count += 1
        # if count == 4:
            # break
        break
id = i
print(encode_dataset[id][1])
print("scores", scores[id].round(3))
print("qrel", qrel_dict[id])
print("retrieved", converted[id])

According to some sources, Disney worried about the rising criminality of the city. A neighboring family had two adolescent children involved in a car barn robbery, and Disney feared that crime would taint his own children. In 1906 he moved with his family to a farm near Marceline, Missouri.
scores [0.725 0.722 0.72  0.717 0.716 0.713 0.713 0.712 0.712 0.712 0.711 0.711
 0.709 0.709 0.709 0.708 0.708 0.707 0.707 0.707]
qrel [4, 7]
retrieved [634, 654, 170, 365, 771, 630, 671, 279, 35, 32, 66, 829, 789, 201, 151, 332, 656, 636, 280, 487]


## BGE

In [422]:
if __name__ == "__main__":
    
    parser = HfArgumentParser((ModelArguments, DataArguments, TrainingArguments))

    model_name = "bge"
    original_model_name = "BAAI/bge-base-en-v1.5"
    root_path = "/home/thuy0050/mg61_scratch2/thuy0050/exp/tevatron/temporal_nobel_prize/temporal"

    cfg = configs[model_name]
    model_name_or_path = os.path.join(root_path, original_model_name, "/home/thuy0050/mg61_scratch2/thuy0050/exp/tevatron/temporal_nobel_prize/temporal/BAAI/bge-base-en-v1.5/matryoshka_64-128-256-512-768_filtered")

    model, tokenizer, encode_loader = get_model_and_tokenizer(cfg,  model_name_or_path)
    
    query = data_args.query_prefix + "Who won the Nobel Prize in Medicine between 1946 and 1947?"
    passage = data_args.passage_prefix + 'Bernardo Houssay\nThe Nobel Prize in Medicine 1947.\nBorn: 10, Apr, 1887, Buenos Aires, Argentina\nDied: 21, Sep, 1971, Buenos Aires, Argentina\nPrize motivation: "for his discovery of the part played by the hormone of the anterior pituitary lobe in the metabolism of sugar"\nAffiliation at the time of the award: Instituto de Biologia y Medicina Experimental (Institute for Biology and Experimental Medicine), Buenos Aires, Argentina\nPrize share: 1/2'
    
    model_output, pooled_output, attention_mask, query_mapping, passage_mapping = get_embedding(query, passage, model_args, data_args, pooling=True)
    
    show_topk_tokens(pooled_output, model_output, attention_mask, query_mapping, query)

Eval with matryoshka dim 768
Topk for dim 64: ['Nobel', 'Prize', 'won', 'Medicine', '1946', '1947', 'in', 'Who', 'and', 'between']
Topk for dim 128: ['Nobel', 'Prize', 'won', 'Who', 'and', '1946', 'between', 'in', '1947', 'the']
Topk for dim 256: ['Nobel', 'Prize', 'won', 'Medicine', 'Who', 'and', 'between', '1946', 'in', 'the']
Topk for dim 512: ['Nobel', 'Prize', 'Medicine', 'between', 'and', 'in', 'won', 'Who', 'the', '1946']
Topk for dim 768: ['Nobel', 'Prize', 'Medicine', 'in', 'won', 'the', 'and', 'between', 'Who', '1946']


In [424]:
if __name__ == "__main__":
    
    parser = HfArgumentParser((ModelArguments, DataArguments, TrainingArguments))

    model_name = "bge"
    original_model_name = "BAAI/bge-base-en-v1.5"
    root_path = "/home/thuy0050/mg61_scratch2/thuy0050/exp/tevatron/temporal_nobel_prize/temporal"

    cfg = configs[model_name]
    model_name_or_path = os.path.join(root_path, original_model_name, "/home/thuy0050/mg61_scratch2/thuy0050/exp/tevatron/temporal_nobel_prize/temporal/BAAI/bge-base-en-v1.5/qt-pt0.1_filtered_add-cls")

    model, tokenizer, encode_loader = get_model_and_tokenizer(cfg,  model_name_or_path)
    
    query = data_args.query_prefix + "Who won the Nobel Prize in Medicine between 1946 and 1947?"
    passage = data_args.passage_prefix + 'Bernardo Houssay\nThe Nobel Prize in Medicine 1947.\nBorn: 10, Apr, 1887, Buenos Aires, Argentina\nDied: 21, Sep, 1971, Buenos Aires, Argentina\nPrize motivation: "for his discovery of the part played by the hormone of the anterior pituitary lobe in the metabolism of sugar"\nAffiliation at the time of the award: Instituto de Biologia y Medicina Experimental (Institute for Biology and Experimental Medicine), Buenos Aires, Argentina\nPrize share: 1/2'
    
    model_output, pooled_output, attention_mask, query_mapping, passage_mapping = get_embedding(query, passage, model_args, data_args, pooling=True)
    
    show_topk_tokens(pooled_output, model_output, attention_mask, query_mapping, query)

Eval with matryoshka dim 768
Topk for dim 64: ['Nobel', 'Prize', 'Medicine', '1946', 'won', '1947', 'Who', 'and', 'between', 'in']
Topk for dim 128: ['Nobel', 'won', 'and', 'Prize', 'Who', 'between', '1946', '1947', 'in', 'the']
Topk for dim 256: ['Nobel', 'between', 'and', 'Prize', 'won', 'Medicine', 'Who', '1946', 'in', '1947']
Topk for dim 512: ['Nobel', 'between', 'Prize', 'and', 'Medicine', 'Who', 'in', 'won', 'the', '1946']
Topk for dim 768: ['Nobel', 'Prize', 'Medicine', 'won', 'in', 'and', 'between', 'Who', 'the', '1946']


## Extract hard negatives

In [265]:
from tevatron.retriever.modeling.dense import DenseModel
# Assuming ModelArguments, DataArguments, and TrainingArguments are defined
parser = HfArgumentParser((ModelArguments, DataArguments, TrainingArguments))

cfg = configs["gte"]
model_name_or_path = "thenlper/gte-base"

# Example: directly supply your arguments as dicts
arg_dict = {
    "model_name_or_path": model_name_or_path,
    "tokenizer_name": None,
    "padding_side": cfg["padding_side"],
    "dataset_path": "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/test/query.jsonl",
    "dataset_name": "LouisDo2108/temporal-nobel-prize",
    "dataset_config": "corpus",
    "encode_is_query": True,
    "lora_name_or_path": model_name_or_path,
    "pooling": cfg['pooling'],
    "normalize": True,
    "bf16": True,
}

model_args, data_args, training_args = parser.parse_dict(
    {**arg_dict}
)
tokenizer = AutoTokenizer.from_pretrained(
    model_args.tokenizer_name if model_args.tokenizer_name else model_args.model_name_or_path,
    cache_dir=model_args.cache_dir
)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

if data_args.padding_side == 'right':
    tokenizer.padding_side = 'right'
else:
    tokenizer.padding_side = 'left'

if training_args.bf16:
    torch_dtype = torch.bfloat16
elif training_args.fp16:
    torch_dtype = torch.float16
else:
    torch_dtype = torch.float32

model = DenseModel.load(
    model_args.model_name_or_path,
    pooling=model_args.pooling,
    normalize=model_args.normalize,
    lora_name_or_path=model_args.lora_name_or_path,
    cache_dir=model_args.cache_dir,
    torch_dtype=torch_dtype,
    attn_implementation=model_args.attn_implementation,
)
model.matryoshka_dim = 768
model = model.to("cuda")
model = model.eval()

Either your model is not a PEFT-model or you are missing the lora_name_or_path argument.


In [203]:
from torch.utils.data import Dataset
from dataclasses import dataclass
from transformers import PreTrainedTokenizer, ProcessorMixin
import logging
import os
import random
from typing import List, Tuple

from datasets import load_dataset, load_from_disk
from PIL import Image
from torch.utils.data import Dataset
from tqdm import tqdm

class EncodeDataset(Dataset):
    """
    Dataset for encoding.
    Loads data and optionally shards it for distributed processing.
    """

    def __init__(self, data_args: DataArguments):
        self.data_args = data_args
        self.encode_data = load_dataset(
            self.data_args.dataset_name,
            self.data_args.dataset_config,
            data_files=self.data_args.dataset_path,
            split=self.data_args.dataset_split,
            cache_dir=self.data_args.dataset_cache_dir,
            num_proc=self.data_args.num_proc,
        )
        if self.data_args.dataset_number_of_shards > 1:
            self.encode_data = self.encode_data.shard(
                num_shards=self.data_args.dataset_number_of_shards,
                index=self.data_args.dataset_shard_index,
            )
            
        self.data_args.passage_prefix = self.data_args.passage_prefix.replace("\\n", "\n").strip()
        if self.data_args.passage_prefix != "":
            self.passage_prefix = self.passage_prefix + " "
            
        self.data_args.query_prefix = self.data_args.query_prefix.replace("\\n", "\n").strip()
        if self.data_args.query_prefix != "":
            self.query_prefix = self.query_prefix + " "

    def __len__(self):
        return len(self.encode_data)

    def __getitem__(self, item):
        content = self.encode_data[item]
        
        if self.data_args.encode_is_query:
            content_id = content.get("query_id", "")
            content_text = content.get('query', content.get('query', ''))
            content_text = self.data_args.query_prefix + content_text.strip()
        else:
            content_id = content.get("query_id", "")
            content_text = content.get('query', content.get('query', ''))
            content_text = self.data_args.passage_prefix + content_text.strip()

        return content_id, content_text


@dataclass
class EncodeCollator:
    """
    simple collator for text only data.
    """
    data_args: DataArguments
    tokenizer: PreTrainedTokenizer

    def __call__(self, features):
        """
        Collate function for encoding.
        :param features: list of (id, text, image) tuples
        but in this case, it's just image is None
        """
        content_ids = [x[0] for x in features]
        texts = [x[1] for x in features]
        max_length = self.data_args.query_max_len if self.data_args.encode_is_query else self.data_args.passage_max_len
        collated_inputs = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=(
                max_length - 1 if self.data_args.append_eos_token else max_length
            ),
            pad_to_multiple_of=self.data_args.pad_to_multiple_of,
            return_tensors="pt",
            return_attention_mask=True,
            return_token_type_ids=False,
            add_special_tokens=True,
            padding_side=self.data_args.padding_side
        )
        return content_ids, collated_inputs



In [383]:
temp = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/train_enhanced_temporal.jsonl", jsonl=True)

corpus = []
corpus_mapping = {}
for ix, item in enumerate(temp):
    corpus.append({
        "id": ix,
        "query_id": item.get("query_id", -1),
        "original_corpus_id": item["positive_passages"][0].get("docid", -1),
        "query": item.get("query", -1),
        "temporal": item.get("temporal", -1),
    })
    corpus_mapping[ix] = item["positive_passages"][0].get("docid", -1)
write_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/hard_negatives/corpus.jsonl", corpus, jsonl=True)

negs = []
count = 0
pos_to_passage_dict = defaultdict(list)
neg_to_passage_dict = defaultdict(list)

for ix, item in enumerate(temp):
    current_id = int(item.get("query_id", -1))
    pos = item.get("positive_passages", [])
    pos_id = pos[0].get("docid", -1)
    neg = item.get("negative_passages", [])
    pos = [(x["text"], x["temporal"]) for x in pos]
    neg = [(x["text"], x["temporal"]) for x in neg]

    for x, t in pos:
        negs.append({
            "id": count,
            "query_id": current_id,
            "original_corpus_id": ix,
            "query": x,
            "type": "positive",
            "temporal": t,
        })
        pos_to_passage_dict[ix].append(count)
        count += 1
        
        
    for x, t in neg:
        negs.append({
            "id": count,
            "query_id": current_id,
            "original_corpus_id": ix,
            "query": x,
            "type": "negative",
            "temporal": t,
        })
        neg_to_passage_dict[ix].append(count)
        count += 1
    
    
write_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/hard_negatives/negs.jsonl", negs, jsonl=True)

The file is of type: <class 'list'>
The file contains 11693 items.
The file contains 11693 items.
Saved to /home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/hard_negatives/corpus.jsonl
The file contains 125532 items.
Saved to /home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/hard_negatives/negs.jsonl


### Negatives side

In [268]:
arg_dict["dataset_path"] = "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/hard_negatives/negs.jsonl"

model_args, data_args, training_args = parser.parse_dict(
    {**arg_dict}
)

encode_dataset = EncodeDataset(
    data_args=data_args,
)

encode_collator = EncodeCollator(
    data_args=data_args,
    tokenizer=tokenizer,
)

encode_loader = DataLoader(
    encode_dataset,
    batch_size=2048,
    collate_fn=encode_collator,
    shuffle=False,
    drop_last=False,
    num_workers=4,
)

print(encode_dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

(2, 'What was the residence of Elias Disney from 1906 to 1910?')


In [269]:
data_args.encode_is_query = True
encoded = []
lookup_indices = []
for (batch_ids, batch) in tqdm(encode_loader):
        lookup_indices.extend(batch_ids)
        with (
            torch.autocast(
                "cuda", dtype=torch.float16 if training_args.fp16 else torch.bfloat16
            )
            if training_args.fp16 or training_args.bf16
            else nullcontext()
        ):
            with torch.no_grad():
                for k, v in batch.items():
                    batch[k] = v.to("cuda")
                if data_args.encode_is_query:
                    model_output: EncoderOutput = model(query=batch)
                    encoded.append(model_output.q_reps.cpu().detach().numpy())
                else:
                    model_output: EncoderOutput = model(passage=batch)
                    encoded.append(model_output.p_reps.cpu().detach().numpy())

encoded = np.concatenate(encoded)

with open(os.path.join("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/hard_negatives", "negs_embed_768.pkl"), "wb") as f:
    pickle.dump((encoded, lookup_indices), f)
                    
with open(os.path.join("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/hard_negatives", "negs_embed_768.pkl"), "rb") as f:
    p_vecs, corpus_lookup_indices = pickle.load(f)

# create a basic flat index with dimension match our embedding
index = faiss.IndexFlatIP(len(p_vecs[0]))
# make sure the embeddings are float32
p_vecs = np.asarray(p_vecs, dtype=np.float32)
# use gpu to accelerate index searching
if torch.cuda.is_available():
    co = faiss.GpuMultipleClonerOptions()
    co.shard = True
    co.useFloat16 = True
    index = faiss.index_cpu_to_all_gpus(index, co=co)
# add all the embeddings to the index
index.add(p_vecs)

  0%|                                                                                                                                                                                                                                        | 0/62 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
To disable this warning, you can either:
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable 	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the en

Eval with matryoshka dim 768


  3%|███████▏                                                                                                                                                                                                                        | 2/62 [00:08<03:30,  3.51s/it]

Eval with matryoshka dim 768
Eval with matryoshka dim 768


  6%|██████████████▍                                                                                                                                                                                                                 | 4/62 [00:08<01:11,  1.23s/it]

...

 94%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌              | 58/62 [00:16<00:00,  6.71it/s]

Eval with matryoshka dim 768
Eval with matryoshka dim 768
Eval with matryoshka dim 768


 98%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▍   | 61/62 [00:16<00:00,  6.84it/s]

Eval with matryoshka dim 768
Eval with matryoshka dim 768


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 62/62 [00:17<00:00,  3.62it/s]


### Corpus side

In [341]:
arg_dict["dataset_path"] = "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/hard_negatives/corpus.jsonl"

model_args, data_args, training_args = parser.parse_dict(
    {**arg_dict}
)

encode_dataset = EncodeDataset(
    data_args=data_args,
)

encode_collator = EncodeCollator(
    data_args=data_args,
    tokenizer=tokenizer,
)

encode_loader = DataLoader(
    encode_dataset,
    batch_size=2048,
    collate_fn=encode_collator,
    shuffle=False,
    drop_last=False,
    num_workers=4,
)
print(encode_dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

(2, 'According to some sources, Disney worried about the rising criminality of the city. A neighboring family had two adolescent children involved in a car barn robbery, and Disney feared that crime would taint his own children. In 1906 he moved with his family to a farm near Marceline, Missouri.')


In [344]:
data_args.encode_is_query = False
encoded = []
lookup_indices = []
for (batch_ids, batch) in tqdm(encode_loader):
        lookup_indices.extend(batch_ids)
        with (
            torch.autocast(
                "cuda", dtype=torch.float16 if training_args.fp16 else torch.bfloat16
            )
            if training_args.fp16 or training_args.bf16
            else nullcontext()
        ):
            with torch.no_grad():
                for k, v in batch.items():
                    batch[k] = v.to("cuda")
                if data_args.encode_is_query:
                    model_output: EncoderOutput = model(query=batch)
                    encoded.append(model_output.q_reps.cpu().detach().numpy())
                else:
                    model_output: EncoderOutput = model(passage=batch)
                    encoded.append(model_output.p_reps.cpu().detach().numpy())
                    
# encoded = np.concatenate(encoded)

# with open(os.path.join("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/hard_negatives", "corpus_embed_768.pkl"), "wb") as f:
#     pickle.dump((encoded, lookup_indices), f)

  0%|                                                                                                                                                                                                                                         | 0/6 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already b

Eval with matryoshka dim 768


 17%|█████████████████████████████████████▌                                                                                                                                                                                           | 1/6 [00:02<00:10,  2.13s/it]

Eval with matryoshka dim 768


 33%|███████████████████████████████████████████████████████████████████████████                                                                                                                                                      | 2/6 [00:02<00:05,  1.28s/it]

...

 67%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████                                                                           | 4/6 [00:04<00:01,  1.05it/s]

Eval with matryoshka dim 768


 83%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌                                     | 5/6 [00:05<00:00,  1.04it/s]

Eval with matryoshka dim 768


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:06<00:00,  1.02s/it]


In [388]:
all_ids = []
all_scores = []
for batch in encoded:
    score, ids = index.search(
        batch, k=30,
    )
    all_ids.extend([x for x in ids[:, 5:]])
    all_scores.extend([x for x in score[:, 5:]])

In [ ]:
for item in temp:
    for pos in item.get("positive_passages", []):
        if "2025" in pos["temporal"]:
            

{'query_id': 2,
 'query': 'According to some sources, Disney worried about the rising criminality of the city. A neighboring family had two adolescent children involved in a car barn robbery, and Disney feared that crime would taint his own children. In 1906 he moved with his family to a farm near Marceline, Missouri.',
 'temporal': ['1906'],
 'positive_passages': [{'docid': 2756,
   'text': 'What was the residence of Elias Disney from 1906 to 1910?',
   'temporal': ['from 1906 to 1910'],
   'temporal_query_type': 'Explicit',
   'allen_relation': 'During'},
  {'docid': 2756,
   'text': 'What was the residence of Elias Disney in October 1906?',
   'temporal': ['in October 1906'],
   'temporal_query_type': 'Explicit',
   'allen_relation': 'Equals'},
  {'docid': 2756,
   'text': 'What was the residence of Elias Disney after moving in 1906?',
   'temporal': ['after moving in 1906'],
   'temporal_query_type': 'Explicit',
   'allen_relation': 'After'},
  {'docid': 2756,
   'text': 'What was 

In [389]:
count = 0

for ix, (score, top_ids) in enumerate(zip(all_scores, all_ids)):
    if ix != 10:
        continue
    current_docid = corpus[ix]["id"]
    original_docid = corpus[ix]["original_corpus_id"]
    doc_text = corpus[ix]["query"]
    
    pos_docids = pos_to_passage_dict[current_docid]
    neg_docids = neg_to_passage_dict[current_docid]
    

    new_hard_neg_list = []
    print(f"Current_docid: {current_docid}; Original_docid: {original_docid}")
    print(f"doc_text: {doc_text}")
    print("length of positives:", len(pos_docids))
    print("length of negatives:", len(neg_docids))

    for top, i in enumerate(tqdm(top_ids)):
        # Get the corresponding docid of the negative
        corresponding_docid = negs[int(i)]["original_corpus_id"] # This is for the convenience of encode_dataset lookup
        corerresponding_original_docid = corpus_mapping[corresponding_docid]
        print(f"Considering top-{top+1} id:", i)
        print("Corresponding original docid:", corerresponding_original_docid)
        
        # if corerresponding_original_docid == original_docid:
        #     print("Skipping same original docid")
        #     continue
        
        if corresponding_docid == current_docid:
            print("Skipping same docid")
            continue
        
        if i in pos_docids:
            print(f"Skipping positive: {negs[int(i)]}")
            continue
        
        # if i in neg_docids:
        #     print(f"Adding negative: {negs[int(i)]}")
        #     new_hard_neg_list.append(i)
        # else:
        print(f"Adding others: {negs[int(i)]}")
        new_hard_neg_list.append(i)
        

    print(len(set(new_hard_neg_list)))

Current_docid: 10; Original_docid: 4157
doc_text: Manager Chris Wilder said in January 2009 he wanted to extend Constables stay the club, claiming he epitomises what I am trying to build here at the club. With regard to extending his stay at Oxford, Constable said he was open to offers.
length of positives: 6
length of negatives: 5


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 25/25 [00:00<00:00, 36714.85it/s]

Considering top-1 id: 4902
Corresponding original docid: 4372
Adding others: {'id': 4902, 'query_id': 300, 'original_corpus_id': 459, 'query': 'How long did the player stay with Oxford?', 'type': 'positive', 'temporal': []}
Considering top-2 id: 56553
Corresponding original docid: 4157
Adding others: {'id': 56553, 'query_id': 3662, 'original_corpus_id': 5280, 'query': 'Which team did James Constable play for after 1 January 2009?', 'type': 'negative', 'temporal': ['after 1 January 2009']}
Considering top-3 id: 56556
Corresponding original docid: 4157
Adding others: {'id': 56556, 'query_id': 3662, 'original_corpus_id': 5281, 'query': 'Which team did James Constable play for after the summer signings in 2025?', 'type': 'positive', 'temporal': ['after the summer signings in 2025']}
Considering top-4 id: 106
Corresponding original docid: 4157
Skipping same docid
Considering top-5 id: 80
Corresponding original docid: 4157
Adding others: {'id': 80, 'query_id': 6, 'original_corpus_id': 7, 'qu